<a href="https://colab.research.google.com/github/JFSS20000/07MIAR04/blob/Actividad_Articulo_C1/Model_ResUnet_JOSE_FERNANDO_SARMIENTO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
# Instala la librería
#!pip install keras-unet-collection
#!pip install --upgrade keras-unet-collection
!pip install tensorflow

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from keras_unet_collection import models

from tensorflow import keras
from keras.preprocessing.image import load_img
from keras import layers


In [10]:
import re
import argparse

from PIL import Image
from io import BytesIO
from bs4 import BeautifulSoup
from skimage import io as skio
from urllib.request import urlopen
import os

def html_url_parser(url, save_dir, show=False, wait=False):
    """
    HTML parser to download images from URL.
    Params:\n
    `url` - Image url\n
    `save_dir` - Directory to save extracted images\n
    `show` - Show downloaded image\n
    `wait` - Press key to continue executing
    """

    website = urlopen(url)
    html = website.read()

    soup = BeautifulSoup(html, "html5lib")

    for image_id, link in enumerate(soup.find_all('a', href=True)):
        if(image_id == 0):
            continue


        img_url = link['href']

        try:
            if os.path.isfile(save_dir + "%d.png" % image_id) == False:
                print("[INFO] Downloading image from URL:", link['href'])
                image = Image.open(urlopen(img_url))
                image.save(save_dir + "%d.png" % image_id, "PNG")
                if(show):
                    image.show()
            else:
                print('skipped')
        except KeyboardInterrupt:
            print("[EXCEPTION] Pressed 'Ctrl+C'")
            break
        except Exception as image_exception:
            print("[EXCEPTION]", image_exception)
            continue

        if(wait):
            key = input("[INFO] Press any key to continue ('q' to exit)... ")
            if(key.lower() == 'q'):
                break

In [14]:

URL_TEST_IMG  = "https://www.cs.toronto.edu/~vmnih/data/mass_roads/test/sat/index.html"
URL_TEST_GT  = "https://www.cs.toronto.edu/~vmnih/data/mass_roads/test/map/index.html"

html_url_parser(url=URL_TEST_IMG, save_dir="./testing/image/")
html_url_parser(url=URL_TEST_GT, save_dir="./testing/mask/")

print("[INFO] All done!")

skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
[INFO] Downloading image from URL: http://www.cs.toronto.edu/~vmnih/data/mass_roads/test/sat/18328780_15.tiff
[INFO] Downloading image from URL: http://www.cs.toronto.edu/~vmnih/data/mass_roads/test/sat/18328960_15.tiff
[INFO] Downloading image from URL: http://www.cs.toronto.edu/~vmnih/data/mass_roads/test/sat/18478735_15.tiff
[INFO] Downloading image from URL: http://www.cs.toronto.edu/~vmnih/data/mass_roads/test/sat/18478900_15.tiff
[INFO] Downloading image from URL: http://www.cs.toronto.edu/~vmnih/data/mass_roads/test/sat/18478930_15.tiff
[INFO] Downloading image from URL: http://www.cs.toronto.edu/~vmnih/data/mass_roads/test/sat/20278885_15.tiff
[INFO] Downloading image from URL: http://www.cs.toronto.edu/~vmnih/data/mass_roads/test/sat/20728960_15.tiff
[INFO] Downloading image from URL: http://www.cs.toronto.edu/~vmnih/data/mass_roads/test/sat/20878930_15.tiff
[INFO] Downloadi

In [15]:
from tensorflow.keras.layers import Conv2D, BatchNormalization, Activation, Add, Conv2DTranspose, concatenate, Lambda, UpSampling2D
from tensorflow.keras import Model, Input
from contextlib import redirect_stdout
import tensorflow as tf

In [17]:

############################# CONVOLUTIONAL BLOCK #############################

def conv_block(feature_map):

    # Main Path
    conv_1 = Conv2D(filters=64, kernel_size=(3,3), strides=(1,1), padding='same')(feature_map)
    bn = BatchNormalization()(conv_1)
    relu = Activation(activation='relu')(bn)
    conv_2 = Conv2D(filters=64, kernel_size=(3,3), strides=(1,1), padding='same')(relu)

    res_conn = Conv2D(filters=64, kernel_size=(1,1), strides=(1,1), padding='same')(feature_map)
    res_conn = BatchNormalization()(res_conn)
    addition = Add()([res_conn, conv_2])

    return addition

############################### RESIDUAL BLOCK ################################

def res_block(feature_map, conv_filter, stride):

    bn_1 = BatchNormalization()(feature_map)
    relu_1 = Activation(activation='relu')(bn_1)
    conv_1 = Conv2D(conv_filter, kernel_size=(3,3), strides=stride[0], padding='same')(relu_1)
    bn_2 = BatchNormalization()(conv_1)
    relu_2 = Activation(activation='relu')(bn_2)
    conv_2 = Conv2D(conv_filter, kernel_size=(3,3), strides=stride[1], padding='same')(relu_2)


    res_conn = Conv2D(conv_filter, kernel_size=(1,1), strides=stride[0], padding='same')(feature_map)
    res_conn = BatchNormalization()(res_conn)
    addition = Add()([res_conn, conv_2])

    return addition

################################### ENCODER ###################################

def encoder(feature_map):

    # Initialize the to_decoder connection
    to_decoder = []

    # Block 1 - Convolution Block
    path = conv_block(feature_map)
    to_decoder.append(path)

    # Block 2 - Residual Block 1
    path = res_block(path, 128, [(2, 2), (1, 1)])
    to_decoder.append(path)

    # Block 3 - Residual Block 2
    path = res_block(path, 256, [(2, 2), (1, 1)])
    to_decoder.append(path)

    return to_decoder

################################### DECODER ###################################

def decoder(feature_map, from_encoder):

    # Block 1: Up-sample, Concatenation + Residual Block 1
    main_path = UpSampling2D(size=(2,2), interpolation='bilinear')(feature_map)
    # main_path = Conv2DTranspose(filters=256, kernel_size=(2,2), strides=(2,2), padding='same')(feature_map)
    main_path = concatenate([main_path, from_encoder[2]], axis=3)
    main_path = res_block(main_path, 256, [(1, 1), (1, 1)])

    # Block 2: Up-sample, Concatenation + Residual Block 2
    main_path = UpSampling2D(size=(2,2), interpolation='bilinear')(main_path)
    # main_path = Conv2DTranspose(filters=128, kernel_size=(2,2), strides=(2,2), padding='same')(main_path)
    main_path = concatenate([main_path, from_encoder[1]], axis=3)
    main_path = res_block(main_path, 128, [(1, 1), (1, 1)])

    # Block 3: Up-sample, Concatenation + Residual Block 3
    main_path = UpSampling2D(size=(2,2), interpolation='bilinear')(main_path)
    # main_path = Conv2DTranspose(filters=64, kernel_size=(2,2), strides=(2,2), padding='same')(main_path)
    main_path = concatenate([main_path, from_encoder[0]], axis=3)
    main_path = res_block(main_path, 64, [(1, 1), (1, 1)])

    return main_path

################################ RESIDUAL UNET ################################

def ResUNet(inputshape):

    # Input
    model_input = Input(shape=inputshape)
    model_input_float = Lambda(lambda x: x / 255)(model_input)

    # Encoder Path
    model_encoder = encoder(model_input_float)

    # Bottleneck
    model_bottleneck = res_block(model_encoder[2], 512, [(2, 2), (1, 1)])

    # Decoder Path
    model_decoder = decoder(model_bottleneck, model_encoder)

    # Output
    model_output = Conv2D(filters=1, kernel_size=(1, 1), strides=(1, 1), activation='sigmoid', padding='same')(model_decoder)

    return Model(model_input, model_output)

################################ SANITY CHECK #################################

# The last couple of lines are only used for a sanity check. It outputs a summary of the layers to verify
# if each layer is outputting the correct dimension as expected. The final line outputs a graphic of
# the actual network for a visual reference

# model = ResUNet((224, 224, 3))
# model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
# model.summary()
# tf.keras.utils.plot_model(model, to_file='model.png', show_layer_names=True, show_shapes=True, rankdir='TB')

In [18]:
model = ResUNet((224, 224, 3))
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()
tf.keras.utils.plot_model(model, to_file='model.png', show_layer_names=True, show_shapes=True, rankdir='TB')

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 224, 224,  │          0 │ input_layer_1[0]… │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 224, 224,  │      1,792 │ lambda[0][0]      │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 224, 224,  │        256 │ conv2d_9[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_11 (Conv2D)  │ (None, 224, 224,  │        256 │ lambda[0][0]      │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_15       │ (None, 224, 224,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 224, 224,  │        256 │ conv2d_11[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_10 (Conv2D)  │ (None, 224, 224,  │     36,928 │ activation_15[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_7 (Add)         │ (None, 224, 224,  │          0 │ batch_normalizat… │
│                     │ 64)               │            │ conv2d_10[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 224, 224,  │        256 │ add_7[0][0]       │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_16       │ (None, 224, 224,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_12 (Conv2D)  │ (None, 112, 112,  │     73,856 │ activation_16[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        512 │ conv2d_12[0][0]   │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_14 (Conv2D)  │ (None, 112, 112,  │      8,320 │ add_7[0][0]       │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_17       │ (None, 112, 112,  │          0 │ batch_normalizat… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        512 │ conv2d_14[0][0]   │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_13 (Conv2D)  │ (None, 112, 112,  │    147,584 │ activation_17[0]

 Total params: 8,233,025 (31.41 MB)

 Trainable params: 8,223,809 (31.37 MB)

 Non-trainable params: 9,216 (36.00 KB)

In [8]:
# Llamar a la función get_model con una imagen de 160x160
img_size = (160, 160)
num_classes = 4
road_model = get_model(img_size, num_classes)
print(road_model.summary())

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 160, 160,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 80, 80,    │        896 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 80, 80,    │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 80, 80,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 80, 80,    │          0 │ activation[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d    │ (None, 80, 80,    │      2,400 │ activation_1[0][… │
│ (SeparableConv2D)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 80, 80,    │        256 │ separable_conv2d… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 80, 80,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d_1  │ (None, 80, 80,    │      4,736 │ activation_2[0][… │
│ (SeparableConv2D)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 80, 80,    │        256 │ separable_conv2d… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 40, 40,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 40, 40,    │      2,112 │ activation[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 40, 40,    │          0 │ max_pooling2d[0]… │
│                     │ 64)               │            │ conv2d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 40, 40,    │          0 │ add[0][0]         │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d_2  │ (None, 40, 40,    │      8,896 │ activation_3[0][… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 40, 40,    │        512 │ separable_conv2d… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_4        │ (None, 40, 40,    │          0 │ batch_normalizat

 Total params: 2,059,268 (7.86 MB)

 Trainable params: 2,055,492 (7.84 MB)

 Non-trainable params: 3,776 (14.75 KB)

None


In [9]:
#WEIGHTS_PATH_160 = 'road_segmentation_160_160.h5'
WEIGHTS_PATH = '/content/Best ResUNet.hdf5'

try:
    # Carga los pesos del archivo .h5
    road_model.load_weights(WEIGHTS_PATH)
    print("✅ Pesos 160x160 cargados exitosamente.")

except FileNotFoundError:
    print(f"❌ Archivo de pesos no encontrado en la ruta: {WEIGHTS_PATH}. Debes descargarlo primero.")
except ValueError:
    print("❌ Error: La arquitectura del modelo (160x160) no coincide con la de los pesos.")
except Exception as e:
    print(f"❌ Ocurrió un error inesperado al cargar los pesos 160x160: {e}")

❌ Error: La arquitectura del modelo (160x160) no coincide con la de los pesos.


In [ ]:
# Definir arquitectura U-Net
#     320x320 píxeles
#     3 canales de entrada
#     1 canal de salida
# (ej. binario: carretera/no carretera)

IMG_SIZE = (320, 320)
NUM_CLASSES = 4

road_model = models.unet_2d(
    #input_size=(320, 320, 3),
    #filter_num=[32, 64, 128, 256, 512],   # Número de filtros por nivel
    #n_labels=1,                           # Una clase de salida (la carretera)
    # stack_num=2,                         # Doble convolución en cada bloque - Removed incorrect parameter
    #depth=5,                              # Use depth to specify the number of pooling/downsampling levels
    input_size=IMG_SIZE + (3,),      # (320, 320, 3)
    filter_num=[32, 64, 128, 256, 512], # Filtros inferidos
    n_labels=NUM_CLASSES,
    activation='ReLU',
    #output_activation='Sigmoid',          # Para segmentación binaria
    output_activation='Softmax',           # Para segmentación multiclase
    #output_padding='same',
    #use_bias=True,
    batch_norm=True,
    pool=True,
    unpool=True,
    name='unet_road_320'
)

# Compilación
road_model.compile(
    #optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    optimizer=tf.keras.optimizers.Adam(),
    #loss='binary_crossentropy',
    loss='categorical_crossentropy' if NUM_CLASSES > 1 else 'binary_crossentropy',
    metrics=['accuracy']
)

# Modelo definido, aún no tiene pesos entrenados.

In [ ]:
WEIGHTS_PATH = 'road_segmentation_320_320.h5'
try:
    # Carga los pesos del archivo .h5
   # keras.models.load_model('res_unet_carreteras.h5')
    model1.load_weights(WEIGHTS_PATH)
    print("✅ Pesos cargados exitosamente.")

except FileNotFoundError:
    print("❌ Archivo de pesos no encontrado. Debes descargarlo primero.")
except ValueError:
    print("❌ Error: La arquitectura del modelo no coincide con la de los pesos.")

❌ Error: La arquitectura del modelo no coincide con la de los pesos.
